# 04 — Attn-LOB Model (architecture wiring check, no training)

Builds the Attn-LOB CNN-Attention network from paper Section III-B3 / Fig. 1 (`paper_replication.models.AttnLOB`) and runs it on a real batch from the feature-engineering pipeline (`paper_replication.features`, notebook 02) built off the small LOB snapshot file.

**This notebook does not train the model.** No optimizer, no loss backward pass over the dataset — just enough forward passes to confirm the architecture is wired correctly end-to-end against real data (shapes, finite outputs, no NaNs). The model's weights are randomly initialized throughout, so predictions here carry no signal.

In [ ]:
from __future__ import annotations

import torch

from paper_replication.features import FeatureConfig, build_feature_dataset
from paper_replication.features.dataset import chronological_split
from paper_replication.models import AttnLOB, AttnLOBConfig

LOB_PATH = "../data/btc_usdt_20221019_20221030_lob.parquet"

torch.manual_seed(0)

## Build the dataset (reusing notebook 02's pipeline, default config)

The pretrain task (paper Table I) only uses the raw LOB state as model input — `dataset.dynamic_state` exists (RV/RSI/OSI) but isn't consumed by `AttnLOB` here; it's reserved for a later RL-phase head.

In [ ]:
feature_config = FeatureConfig()
dataset = build_feature_dataset(LOB_PATH, feature_config)
splits = chronological_split(dataset)

print(f"n_samples total = {len(dataset)}")
print(f"lob_state shape  = {dataset.lob_state.shape}")
for name, split in splits.items():
    print(f"  {name:>5}: n={len(split)}")

## Instantiate the model

`AttnLOBConfig` defaults match the paper's n_levels=10 setup (`4*n_levels=40` input width, the conv stack's kernel/stride sequence is fixed to this).

In [ ]:
model_config = AttnLOBConfig()
model = AttnLOB(model_config)
model.eval()  # no dropout/batchnorm-update side effects for this sanity check

n_params = sum(p.numel() for p in model.parameters())
print(model_config)
print(f"embed_dim  = {model_config.embed_dim}")
print(f"n_params   = {n_params:,}")

## Forward pass on a real batch (no gradient, no training)

Pulls a batch straight from the built dataset's `lob_state` array and runs it through the model once.

In [ ]:
batch_size = 16
test_split = splits["test"]

x = torch.from_numpy(test_split.lob_state[:batch_size]).float()
y_true = test_split.labels[:batch_size]

with torch.no_grad():
    logits = model(x)
    probs = torch.softmax(logits, dim=-1)
    pooled = model.forward_features(x)

print(f"input   shape = {tuple(x.shape)}")
print(f"logits  shape = {tuple(logits.shape)}")
print(
    f"pooled  shape = {tuple(pooled.shape)}  (192-dim backbone repr, for a future RL head)"
)
print(f"all finite    = {bool(torch.isfinite(logits).all())}")

In [ ]:
label_names = {-1: "down", 0: "stationary", 1: "up"}
# Convention only (not learned): output index 0/1/2 <-> label -1/0/1, ascending.
# This mapping matters once we train (e.g. target = label + 1 for cross_entropy)
# but right now the model is random-initialized, so these probs carry no signal.
pred_class = probs.argmax(dim=-1).numpy() - 1

for i in range(batch_size):
    true_name = label_names[int(y_true[i])]
    pred_name = label_names[int(pred_class[i])]
    p = probs[i].tolist()
    print(
        f"true={true_name:>10}  pred={pred_name:>10}  probs(down/stat/up)={[f'{v:.2f}' for v in p]}"
    )

## Summary

The architecture is wired correctly end-to-end against real data: `(batch, 50, 40)` LOB-state windows produce finite `(batch, 3)` pretrain logits and a `(batch, 192)` pooled backbone representation, matching paper Table I's Attn-LOB input/output shapes.

The predictions above consistently favor one class rather than looking uniform -- expected here, not a bug: in `eval()` mode the `BatchNorm2d` layers use their randomly-initialized running statistics (never updated, since nothing has trained), so the whole batch gets pushed through the same fixed, untrained bias. It says nothing about the market; it's an artifact of never having trained. This notebook stops here. Training (an actual optimizer loop over `splits["train"]`/`splits["val"]`, mirroring Table I's precision/recall/F1 comparison) is separate, future work.